# Tabular Data Augmentation and Classification

This notebook uses a tabular classification dataset and compares:
- baseline training
- simple tabular augmentation with Gaussian noise on the training set

This is a beginner-friendly example.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.keras.utils.set_random_seed(42)

In [ ]:
data = load_breast_cancer()
X = data.data.astype(np.float32)
y = data.target.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

X_noise = X_train[:200] + np.random.normal(0, 0.05, size=X_train[:200].shape).astype(np.float32)
y_noise = y_train[:200]

X_train_aug = np.concatenate([X_train, X_noise], axis=0)
y_train_aug = np.concatenate([y_train, y_noise], axis=0)

In [ ]:
def build_tabular_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

base_model = build_tabular_model(X_train.shape[1])
base_model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
base_acc = base_model.evaluate(X_test, y_test, verbose=0)[1]

aug_model = build_tabular_model(X_train.shape[1])
aug_model.fit(X_train_aug, y_train_aug, epochs=20, batch_size=32, verbose=0)
aug_acc = aug_model.evaluate(X_test, y_test, verbose=0)[1]

print("Baseline accuracy:", round(base_acc, 4))
print("Augmented accuracy:", round(aug_acc, 4))